### SoRL Warm-up: Clustering-based SFT → Full SoRL

**Pipeline**:
1. Extract K-chunk embeddings from training data using pretrained model
2. K-means → 128 cluster centroids = abstract codebook
3. Initialize abstract embeddings with centroids
4. SFT warmup: train model to predict labeler-assigned abstract token IDs
5. Switch to full SoRL v3 training

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from data.pt_dataset import get_dataset, evaluate_accuracy, collate_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Using device: cpu


In [2]:
# ---- Config ----
MODEL_NAME = "Qwen/Qwen3-0.6B"
ABS_VOCAB  = 128
K          = 4
DATASET    = "gsm8k"
MAX_LENGTH = 256
BATCH_SIZE = 2
N_CHUNKS_FOR_CLUSTERING = 50000  # how many K-chunks to collect for K-means

In [3]:
# ---- Load model + data ----
model = SorlModelWrapper.from_pretrained(MODEL_NAME, abstract_vocab_size_list=[ABS_VOCAB])
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_vocab = int(model.vocab_sizes[0].item())

train_ds = get_dataset(DATASET, split="train", tokenizer=tokenizer, max_length=MAX_LENGTH)
val_ds   = get_dataset(DATASET, split="test",  tokenizer=tokenizer, max_length=MAX_LENGTH)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | base_vocab: {base_vocab}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Train: 7473 | Val: 1319 | base_vocab: 151936


In [4]:
# ============================================================
# Step 1: Extract K-chunk embeddings from training data
# ============================================================
# For each sample, take non-overlapping K-chunks of NL tokens
# starting at positions 0, K, 2K, ... and compute mean embedding.

embed_layer = model.model.model.embed_tokens
all_chunk_embs = []

model.eval()
with torch.no_grad():
    for idx in range(len(train_ds)):
        item = train_ds[idx]
        ids = item["input_ids"].to(device)       # (L,)
        mask = item["attention_mask"].to(device)  # (L,)
        pl = item["prompt_len"]
        
        # Only use response tokens (after prompt)
        resp_ids = ids[pl:]
        resp_mask = mask[pl:]
        valid_len = resp_mask.sum().item()
        if valid_len < K:
            continue
        
        resp_ids = resp_ids[:valid_len]
        embs = embed_layer(resp_ids)  # (valid_len, D)
        
        # Non-overlapping K-chunks: [0:K], [K:2K], ...
        n_chunks = valid_len // K
        if n_chunks == 0:
            continue
        truncated = embs[:n_chunks * K].view(n_chunks, K, -1)  # (n_chunks, K, D)
        chunk_means = truncated.mean(dim=1)  # (n_chunks, D)
        all_chunk_embs.append(chunk_means.cpu().float())
        
        if sum(e.shape[0] for e in all_chunk_embs) >= N_CHUNKS_FOR_CLUSTERING:
            break

all_chunk_embs = torch.cat(all_chunk_embs, dim=0).numpy()
print(f"Collected {all_chunk_embs.shape[0]} K-chunk embeddings, dim={all_chunk_embs.shape[1]}")

Collected 50027 K-chunk embeddings, dim=1024


In [ ]:
# ============================================================
# Step 2: K-means clustering → abstract codebook
# ============================================================
# ABS_VOCAB includes the [Mask] token at base_vocab, so we only
# cluster into ABS_VOCAB-1 predictable abstract tokens.
N_CLUSTERS = ABS_VOCAB - 1
kmeans = MiniBatchKMeans(n_clusters=N_CLUSTERS, batch_size=1024, n_init=3, random_state=42)
kmeans.fit(all_chunk_embs)
centroids = torch.from_numpy(kmeans.cluster_centers_).float()  # (N_CLUSTERS, D)

print(f"K-means done. Centroids shape: {centroids.shape} (ABS_VOCAB-1 = {N_CLUSTERS})")
print(f"Inertia: {kmeans.inertia_:.2f}")

# Check cluster distribution
labels = kmeans.labels_
counts = np.bincount(labels, minlength=N_CLUSTERS)
print(f"Cluster sizes — min: {counts.min()}, max: {counts.max()}, median: {np.median(counts):.0f}")

K-means done. Centroids shape: torch.Size([127, 1024]) (ABS_VOCAB-1 = 127)
Inertia: 3598.15
Cluster sizes — min: 22, max: 1206, median: 377


In [6]:
# ============================================================
# Step 3: Initialize abstract embeddings with centroids
# ============================================================
# base_vocab = [Mask] token, predictable abstracts start at base_vocab+1
with torch.no_grad():
    embed_w = model.model.model.embed_tokens.weight
    lm_head_w = model.model.lm_head.weight
    
    centroids_dev = centroids.to(embed_w.device)
    embed_w[base_vocab + 1 : base_vocab + 1 + N_CLUSTERS] = centroids_dev
    lm_head_w[base_vocab + 1 : base_vocab + 1 + N_CLUSTERS] = centroids_dev

print(f"Initialized embed/lm_head[{base_vocab+1}:{base_vocab+1+N_CLUSTERS}] with centroids (skipping [Mask] at {base_vocab})")

# Verify: cosine sim between centroid-initialized embeddings
abs_embs = embed_w[base_vocab + 1 : base_vocab + 1 + N_CLUSTERS].detach()
cos = F.cosine_similarity(abs_embs.unsqueeze(0), abs_embs.unsqueeze(1), dim=-1)
off_diag = cos[~torch.eye(N_CLUSTERS, dtype=bool, device=cos.device)]
print(f"Abstract emb pairwise cos sim — mean: {off_diag.mean():.4f}, max: {off_diag.max():.4f}, min: {off_diag.min():.4f}")

Initialized embed/lm_head[151937:152064] with centroids (skipping [Mask] at 151936)
Abstract emb pairwise cos sim — mean: 0.4961, max: 0.9844, min: -0.0586


In [8]:
# ============================================================
# Labeler: assigns abstract token IDs to K-chunks
# ============================================================

def label_chunks(token_ids, prompt_len, embed_layer, centroids_dev, base_vocab, K):
    """Given NL token_ids (1D), assign abstract token IDs for each K-chunk after prompt.
    Returns: labeled_ids (1D) — original NL tokens with abstract tokens inserted at every K-th position.
    Abstract IDs start at base_vocab+1 (base_vocab is the [Mask] token, not a prediction target).
    """
    resp = token_ids[prompt_len:]  # response portion
    resp_len = len(resp)
    if resp_len < K:
        # Too short, just insert placeholder abstract tokens
        return _insert_placeholders(token_ids, prompt_len, base_vocab, K)
    
    with torch.no_grad():
        embs = embed_layer(resp).float()  # (resp_len, D), cast to float32
    
    # Build expanded sequence: insert abstract token before every K-th response token
    result = list(token_ids[:prompt_len].tolist())
    
    for i in range(0, resp_len, K):
        chunk_end = min(i + K, resp_len)
        chunk_emb = embs[i:chunk_end].mean(dim=0, keepdim=True)  # (1, D)
        
        # Nearest centroid → abstract ID offset by 1 (skip [Mask] at base_vocab)
        sims = F.cosine_similarity(chunk_emb, centroids_dev, dim=-1)  # (N_CLUSTERS,)
        abs_id = base_vocab + 1 + sims.argmax().item()
        
        # Insert abstract token, then the NL chunk
        if i > 0:  # don't insert before first response token
            result.append(abs_id)
        result.extend(resp[i:chunk_end].tolist())
    
    return torch.tensor(result, device=token_ids.device, dtype=token_ids.dtype)


def _insert_placeholders(token_ids, prompt_len, base_vocab, K):
    """Insert abstract tokens at every K-th position (random ID) for short sequences."""
    result = list(token_ids[:prompt_len].tolist())
    resp = token_ids[prompt_len:].tolist()
    for i, t in enumerate(resp):
        if i > 0 and i % K == 0:
            result.append(base_vocab + 1)  # placeholder (first real abstract token)
        result.append(t)
    return torch.tensor(result, device=token_ids.device, dtype=token_ids.dtype)


# Quick test
centroids_dev = centroids.to(device)
item = train_ds[0]
test_labeled = label_chunks(item["input_ids"].to(device), item["prompt_len"], embed_layer, centroids_dev, base_vocab, K)
n_abs = (test_labeled > base_vocab).sum().item()
print(f"Original len: {len(item['input_ids'])} → Labeled len: {len(test_labeled)} | Abstract tokens inserted: {n_abs}")

Original len: 256 → Labeled len: 309 | Abstract tokens inserted: 53


In [9]:
# the label chunks would just insert abstract tokens inside, making the sequence longer
# then we need to do batch level padding again, in case some sequence contains more abstract tokens than others, whilst having less padding tokens


In [ ]:
# ============================================================
# Step 4: SFT Warmup — train jacobi_loss ONLY
# ============================================================
# Sanity check: can the model reconstruct the labeler's abstractions 
# from the [Mask] tokens in a single parallel pass?
import time

SFT_STEPS    = 500
SFT_LR       = 1e-4
SFT_EMB_MULT = 10.0
ALPHA_ABS    = 0.0  # Disabled for this test
ALPHA_TRAJ   = 0.0  # Disabled for this test
ALPHA_JACOBI = 1.0  # Enabled
LOG_EVERY    = 20

# Optimizer: higher LR for embeddings during SFT
emb_params, other_params = [], []
for name, p in model.named_parameters():
    if "embed_tokens" in name or "lm_head" in name:
        emb_params.append(p)
    else:
        other_params.append(p)

sft_optimizer = torch.optim.AdamW([
    {"params": other_params, "lr": SFT_LR},
    {"params": emb_params,   "lr": SFT_LR * SFT_EMB_MULT},
], weight_decay=0.01)

dl = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
dl_iter = iter(dl)

model.train()
sft_history = {"step": [], "abs_loss": [], "traj_loss": [], "jacobi_loss": [], "loss": []}
t0 = time.time()

for step in range(1, SFT_STEPS + 1):
    try:
        batch = next(dl_iter)
    except StopIteration:
        dl_iter = iter(dl)
        batch = next(dl_iter)
    
    input_ids = batch["input_ids"].to(device)
    attn_mask = batch["attention_mask"].to(device)
    prompt_lens = batch["prompt_len"].to(device)
    
    # Label each sample in batch, track prompt lengths
    labeled_seqs = []
    exp_prompt_lens = []
    for b in range(input_ids.shape[0]):
        valid_len = attn_mask[b].sum().item()
        ids_b = input_ids[b, :valid_len]
        pl = prompt_lens[b].item()
        labeled = label_chunks(ids_b, pl, embed_layer, centroids_dev, base_vocab, K)
        labeled_seqs.append(labeled)
        exp_prompt_lens.append(pl)
    
    # Pad to same length
    max_len = max(s.shape[0] for s in labeled_seqs)
    pad_id = tokenizer.pad_token_id
    padded = torch.full((len(labeled_seqs), max_len), pad_id, device=device, dtype=torch.long)
    exp_attn = torch.zeros(len(labeled_seqs), max_len, device=device, dtype=torch.long)
    for b, s in enumerate(labeled_seqs): 
        padded[b, :s.shape[0]] = s
        exp_attn[b, :s.shape[0]] = 1
        
    targets = padded[:, 1:].contiguous()            # (B, L-1)
    is_abs = (targets > base_vocab)  # abstract token positions
    
    abs_loss = torch.tensor(0.0, device=device)
    traj_loss = torch.tensor(0.0, device=device)
        
    # --- 2. Jacobi Forward Pass ---
    # Replace all abstract tokens with the [Mask] token (base_vocab)
    padded_jacobi = padded.clone()
    padded_jacobi[padded_jacobi > base_vocab] = base_vocab
    
    out_jacobi = model(input_ids=padded_jacobi, attention_mask=exp_attn, memory_span_abs=1792, memory_span_traj=1792)
    logits_jacobi = out_jacobi.logits[..., :-1, :].contiguous()
    
    if is_abs.any():
        jacobi_logits = logits_jacobi.clone()
        jacobi_logits[..., :(base_vocab + 1)] = -float("inf")
        safe_abs_targets = targets.clone()
        safe_abs_targets[~is_abs] = base_vocab + 1
        per_tok_jacobi = F.cross_entropy(jacobi_logits.view(-1, jacobi_logits.size(-1)), safe_abs_targets.view(-1), reduction='none')
        per_tok_jacobi = per_tok_jacobi.view(targets.shape) * is_abs.float()
        jacobi_loss = per_tok_jacobi.sum() / is_abs.float().sum().clamp(min=1)
    else:
        jacobi_loss = torch.tensor(0.0, device=device)
    
    # Total loss
    loss = ALPHA_ABS * abs_loss + ALPHA_TRAJ * traj_loss + ALPHA_JACOBI * jacobi_loss
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    sft_optimizer.step()
    sft_optimizer.zero_grad(set_to_none=True)
    
    if step % LOG_EVERY == 0:
        elapsed = time.time() - t0
        print(f"[SFT] step {step:3d}/{SFT_STEPS} | loss={loss.item():.4f} | jacobi={jacobi_loss.item():.4f} | {elapsed:.1f}s")
        sft_history["step"].append(step)
        sft_history["abs_loss"].append(abs_loss.item())
        sft_history["traj_loss"].append(traj_loss.item())
        sft_history["jacobi_loss"].append(jacobi_loss.item())
        sft_history["loss"].append(loss.item())

print(f"SFT warmup done in {time.time()-t0:.1f}s")

In [ ]:
# Plot SFT warmup losses
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, key, title in zip(axes, ["loss", "abs_loss", "traj_loss", "jacobi_loss"], ["Total Loss", "Abstract Loss (AR)", "Traj Loss (NL|abs)", "Jacobi Loss (Masked)"]):
    ax.plot(sft_history["step"], sft_history[key])
    ax.set_xlabel("step")
    ax.set_ylabel(key)
    ax.set_title(f"SFT Warmup: {title}")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Validation: Jacobi Reconstruction Accuracy
# ============================================================
# Can the model correctly guess the labeler's abstract tokens in one shot?
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for i in range(min(50, len(val_ds))):
        item = val_ds[i]
        ids = item["input_ids"].to(device)
        mask = item["attention_mask"].to(device)
        pl = item["prompt_len"]
        
        valid_len = mask.sum().item()
        ids = ids[:valid_len]
        
        # 1. Get the golden labels from the labeler
        labeled = label_chunks(ids, pl, embed_layer, centroids_dev, base_vocab, K)
        
        # 2. Prepare the Jacobi input (replace abstracts with [Mask])
        jacobi_input = labeled.clone()
        is_abs = (jacobi_input > base_vocab)
        jacobi_input[is_abs] = base_vocab
        
        # 3. Forward pass
        out = model(input_ids=jacobi_input.unsqueeze(0), attention_mask=torch.ones_like(jacobi_input).unsqueeze(0))
        logits = out.logits[0, :-1, :]  # (L-1, V)
        targets = labeled[1:]           # (L-1,)
        abs_mask = is_abs[1:]           # (L-1,)
        
        if abs_mask.any():
            # Mask out non-abstract predictions
            abs_logits = logits.clone()
            abs_logits[:, :(base_vocab + 1)] = -float("inf")
            
            # Predictions on abstract positions
            preds = abs_logits[abs_mask].argmax(dim=-1)
            truths = targets[abs_mask]
            
            correct += (preds == truths).sum().item()
            total += abs_mask.sum().item()

print(f"Jacobi Zero-Shot Reconstruction Accuracy: {correct}/{total} ({correct/total*100:.2f}%)")
model.train()
print()

In [ ]:
# ============================================================
# Step 5: Full SoRL v3 training (using the warmed-up model)
# ============================================================
from sorl.trainer_ablate import SoRLTrainerv3, SoRLConfig

config = SoRLConfig(
    num_rollouts=4,
    K=K,
    max_iterations=2,
    memory_span_abs=1792,
    memory_span_traj=1792,
    temperature=1.0,
    alpha_traj=1.0,
    alpha_contrastive=1.0,
    gamma_contrastive=0.5,
    corrupt_method="shuffle",
    corrupt_ratio=0.3,
    alpha_abs=0.5,
    alpha_soft_zipf=2.0,
    alpha_ortho=0.0,
    alpha_anchor=1.0,
    lr=1e-5,
    emb_lr_mult=10.0,
    weight_decay=0.01,
    warmup_steps=50,
    cooldown_frac=0.4,
    max_grad_norm=1.0,
    batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    num_epochs=3,
    log_every=10,
    eval_every=99999,
    save_every=99999,
    output_dir="./ckpt/v3_sft_warmup",
)

trainer = SoRLTrainerv3(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    val_dataset=val_ds,
    config=config,
    device=device,
)

trainer.train()

In [ ]:
# Plot full training loss curves
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("SFT Warmup → SoRL v3 Training Curves", fontsize=14)

plots = [
    ("loss", "Total Loss"),
    ("base_traj_loss", "Base Traj Loss"),
    ("traj_loss", "Traj Loss p(s|a)"),
    ("contrastive_loss", "Hinge Contrastive"),
    ("abs_loss", "Abstract Loss p(a|s)"),
    ("anchor_loss", "Anchor Loss"),
]
for ax, (key, title) in zip(axes.flat, plots):
    if key in trainer.history and len(trainer.history[key]) > 0:
        ax.plot(trainer.history[key], linewidth=0.8)
        ax.set_title(title)
        ax.set_xlabel("log step")
        ax.grid(True, alpha=0.3)
    else:
        ax.set_title(f"{title} (no data)")
plt.tight_layout()
plt.show()

In [ ]:
# Eval
model.eval()
res = evaluate_accuracy(model, tokenizer, val_ds, device, 100)
print(res)

In [ ]:
# ============================================================
# Inner Monologue Visualization
# ============================================================
from data.pt_dataset import _filter_traj_tokens

model.eval()
n_samples = 3
max_new_tokens = 128

for i in range(min(n_samples, len(val_ds))):
    item = val_ds[i]
    input_ids = item["input_ids"].unsqueeze(0).to(device)
    prompt_len = item["prompt_len"]

    question = tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)

    with torch.no_grad():
        generated = model.generate(
            input_ids=input_ids[:, :prompt_len],
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            K=K,
        )

    # Annotated string: NL as text, abstract as [A_id]
    gen_tokens = generated[0, prompt_len:]
    parts = []
    for tok_id in gen_tokens:
        tid = tok_id.item()
        if tid == tokenizer.eos_token_id:
            break
        if tid >= base_vocab:
            parts.append(f"[A{tid - base_vocab}]")
        else:
            parts.append(tokenizer.decode([tid]))

    annotated = "".join(parts)

    # NL-only for comparison
    traj_tokens = _filter_traj_tokens(generated, base_vocab)
    nl_text = tokenizer.decode(traj_tokens[0][prompt_len:], skip_special_tokens=True)

    print(f"{'='*80}")
    print(f"Sample {i+1}")
    print(f"{'='*80}")
    print(f"Question: {question[:200]}...")
    print(f"\n--- Inner Monologue (with abstract tokens) ---")
    print(annotated[:500])
    print(f"\n--- NL-only output ---")
    print(nl_text[:500])
    print()

model.train()
print("Done.")